---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: Fine-tuning

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our previous lectures, we built AI systems using off-the-shelf embeddings and models. One problem with off-the-shelf models is that they can often miss facts about our users and our specific domain. This in turn makes them less-than-optimal for our use case. 

Today, we will see how **fine-tuning** can allow us to circumvent this problem, by adapting a pre-trained model to our specific data. 

Fine-tuned models have several advantages:
1. Much better accuracy on our specific tasks
2. Lower running costs through using smaller, more efficient models
3. Better at handling our domain-specific language and context

Fine-tuning involves (i) creating training data and (ii) running the fine-tuning process.  This demands more upfront work, but it is typically worth it.

*Note: Many teams do not fine-tune their models because they think they need a lot of data or complex setups. This is not always true: even with 100-200 carefully created examples, we can fine-tune a model that has a good shot at outperforming general-purpose models.*

Let's begin with importing the libraries we will use.

In [ ]:
import json
import random
import os
import asyncio
from collections import defaultdict
from textwrap import dedent
from typing import Optional

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv("../.env")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pydantic import BaseModel, Field, field_validator, ValidationInfo
from openai import AsyncOpenAI
import instructor

from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import BatchSemiHardTripletLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import TripletEvaluator
from sentence_transformers.cross_encoder import CrossEncoder
from datasets import Dataset

import lancedb
from lancedb.embeddings import get_registry
from lancedb.pydantic import LanceModel, Vector

from helpers import calculate_mrr, get_recall

random.seed(42)
print("All imports successful!")

# 1. The Business Problem
---

This notebook is inspired by a real problem at [Ramp](https://ramp.com), a corporate card and expense management company.

**The problem** 
- When employees make purchases, each transaction needs to be assigned to a GL (General Ledger) code — an accounting category like "Software & Licenses" or "Travel: Airfare". This is called **account coding**.
- This can be a very difficult task: There's often very little information and hundreds of GL codes Furthermore, some companies even have company-specific codes and rules.

**The retrieval approach**: 
- Rather than building a classifier over a fixed set of categories, Ramp treats this as a **retrieval problem**
- When a new transaction arrives, the system searches a corpus of previously-coded transactions, finds the most similar ones, and suggests their GL codes. 
- This is more flexible than classification — it handles new GL codes naturally, and provides explainable suggestions ("this transaction is similar to these past ones").
- You can read more about the specific problem and what Ramp did to solve it [here](https://engineering.ramp.com/post/transaction-embeddings). Please go after this after the lecture.
  
**Our simplification**: 
- For this lecture, we will take on a simplified version of Ramp's account coding problem. 
- Instead of retrieving from a large corpus of historical transactions, we'll match transactions directly to **category names**. This is effectively a classification-as-retrieval task with 27 fixed categories. 
- The techniques we learn (fine-tuning embeddings, reranking, fine-tuning rerankers) transfer directly to the full retrieval setting — we're just working with a smaller, cleaner version to focus on the methods.

# 2. Dataset
---

Fine-tuning requires **labeled data** from your domain. For our transaction categorization problem, we need pairs of (transaction data, correct expense category).

In practice, you may not have a large labeled dataset to start with. 

To circumvent this problem, a powerful bootstrapping strategy is to just use an LLM to generate synthetic training data. The idea is straightforward:
1. Get some data on how your categories are defined, and what they look like (we already have that in `data/ramp-categories.json`)
2. Prompt an LLM to generate realistic transactions for each category
3. Iteratively refine: review outputs, use good examples as few-shots, generate harder ones

Let's walk through this process.

In [ ]:
# Load categories — expense categories with sample transactions
with open("../data/ramp-categories.json") as f:
    categories = json.load(f)

print(f"Number of categories: {len(categories)}")

print("\nExample category:")
print(json.dumps(categories[0], indent=2))

print("\nAnother example category:")
print(json.dumps(categories[1], indent=2))


Now let's see how our transactions look like by defining a Pydantic model for them.

In [ ]:
# Define a structured representation for transactions
class Transaction(BaseModel):
    merchant_name: str = Field(description="The vendor or service provider's name")
    merchant_category: list[str] = Field(description="General category of the merchant (e.g., Restaurants, SaaS). Should NOT reference the expense category directly.")
    department: str = Field(description="The company department responsible for the transaction")
    location: str = Field(description="Where the transaction took place (city, state/country)")
    amount: float = Field(description="The transaction's monetary value (use non-round numbers, e.g. 1247.83)")
    spend_program_name: str = Field(description="Specific budget or spending authority allocated (like a virtual card name), not the category itself")
    trip_name: Optional[str] = Field(default=None, description="Name of the business trip, if the transaction occurred during travel")
    expense_category: str = Field(description="The GL code / expense category this transaction should be filed under")

    def format_transaction(self):
        """Format as the text representation used in our dataset."""
        return dedent(f"""
        Name : {self.merchant_name}
        Category: {", ".join(self.merchant_category)}
        Department: {self.department}
        Location: {self.location}
        Amount: {self.amount}
        Card: {self.spend_program_name}
        Trip Name: {self.trip_name if self.trip_name else "unknown"}
        """)

    @field_validator("expense_category")
    @classmethod
    def validate_expense_category(cls, v, info: ValidationInfo):
        """Ensure the generated transaction matches the requested category."""
        if not info.context:
            return v
        expected = info.context["category"]["category"]
        assert v == expected, f"Expected category '{expected}', got '{v}'"
        return v

We will use this pydantic model to generate our synthetic data.

In [ ]:
# We'll use instructor instead of litellm -- for no particular reason, just to remind you it exists
client = instructor.from_openai(AsyncOpenAI())
model = "gpt-5.1"

async def generate_transaction(category: dict) -> Transaction:
    """Generate a single synthetic transaction for the given category."""
    return await client.chat.completions.create(
        model=model,
        messages=[{
            "role": "system",
            "content": f"""Generate a realistic transaction for a tech company that would be filed under the category "{category['category']}"."""
        }],
        context={"category": category},
        response_model=Transaction,
    )

In [ ]:
# Generate 5 sample transactions for 5 random categories
sample_categories = random.sample(categories, 5)
transactions = await asyncio.gather(*[generate_transaction(cat) for cat in sample_categories])

# Show one generated transaction
print("Generated transaction object:")
print(transactions[0])
print("\nFormatted (as it appears in our dataset):")
print(transactions[0].format_transaction())
print(f"Label: {transactions[0].expense_category}")

### Iterative refinement

The simple prompt above generates reasonable transactions, but they tend to be too easy to classify — the merchant name and category often give away the answer. To create a more challenging (and realistic) dataset, we will use a two-stage process:

1. **Generate initial batch** with a simple prompt (as above)
2. **Manually review** — keep good examples, edit or reject bad ones
3. **Use approved examples as few-shots** in a harder prompt that asks for **ambiguous** transactions — ones that could plausibly fit multiple categories

This iterative approach produces much better training data because each round of generation learns from human feedback on what makes a good example.

In [ ]:
async def generate_hard_transaction(category: dict, examples: list[Transaction]) -> Transaction:
    """Generate an ambiguous transaction using few-shot examples for harder cases."""
    all_category_names = "\n".join(f"- {cat['category']}" for cat in categories)
    example_json = "\n".join(ex.model_dump_json() for ex in examples)

    return await client.chat.completions.create(
        model="gpt-5.1",
        messages=[{
            "role": "system",
            "content": f"""
            Generate a potentially ambiguous business transaction that could reasonably be categorized as "{category['category']}" or another similar category. The goal is to create transactions that challenge automatic categorization systems.
            
            Available categories:
            {all_category_names}
            
            The transaction should:
            1. Use a realistic merchant name (international names welcome)
            2. Include a non-rounded amount with decimals (e.g., $1247.83)
            3. Be difficult to categorize definitively (could fit multiple categories)
            4. Merchant category should NOT reference the expense category directly
            
            Here are examples of good transactions generated for other categories:
            {example_json}
            """
        }],
        context={"category": category},
        response_model=Transaction,
    )


# Generate 2 harder transactions using our initial batch as few-shots
hard_txns = await asyncio.gather(*[
    generate_hard_transaction(random.choice(categories), transactions)
    for _ in range(2)
])

print("Hard transaction example:")
print(hard_txns[0].format_transaction())
print(f"Label: {hard_txns[0].expense_category}")

### Pre-built dataset

Using this iterative process (generate → review → refine → repeat), we built two pre-built datasets:
- **"small"**: 326 transactions (260 train / 66 eval) — I've reviewed this manually
- **"large"**: 2000 transactions (1600 train / 400 eval) — generated with `gpt-5.1`

Set `DATASET` below to choose which one to use, or set it to `"generate"` to build a fresh dataset from scratch.

In [ ]:
# Choose dataset: "small" (326), "large" (2000), or "generate" (build from scratch)
DATASET = "large"

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

if DATASET == "small":
    train_data = load_jsonl("../data/ramp-train.jsonl")
    eval_data = load_jsonl("../data/ramp-eval.jsonl")

elif DATASET == "large":
    train_data = load_jsonl("../data/ramp-train-2k.jsonl")
    eval_data = load_jsonl("../data/ramp-eval-2k.jsonl")

elif DATASET == "generate":
    # Generate a fresh dataset using the functions defined above
    N_PER_CATEGORY = 20  # ~540 total transactions
    TRAIN_RATIO = 0.8

    print(f"Generating {N_PER_CATEGORY} transactions per category ({N_PER_CATEGORY * len(categories)} total)...")

    # Phase 1: simple generation for all categories
    all_txns = []
    for cat in categories:
        batch = await asyncio.gather(*[generate_transaction(cat) for _ in range(N_PER_CATEGORY)])
        all_txns.extend([t for t in batch if t is not None])
    print(f"Generated {len(all_txns)} transactions")

    # Shuffle and split
    random.shuffle(all_txns)
    split_idx = int(len(all_txns) * TRAIN_RATIO)
    train_txns = all_txns[:split_idx]
    eval_txns = all_txns[split_idx:]

    # Convert to the same dict format as the JSONL files
    train_data = [{"input": t.format_transaction(), "expected": [t.expense_category]} for t in train_txns]
    eval_data = [{"input": t.format_transaction(), "expected": [t.expense_category]} for t in eval_txns]

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Use 'small', 'large', or 'generate'.")

print(f"Dataset: {DATASET}")
print(f"Training transactions: {len(train_data)}")
print(f"Evaluation transactions: {len(eval_data)}")

In [ ]:
# Look at a sample transaction
sample = train_data[0]
print("Sample transaction:\n")
print("**INPUT**")
print(f"{sample['input']}")
print("**EXPECTED CATEGORY**")
print(f"{sample['expected']}")

# How many unique categories are represented in each split?
train_categories = set(item["expected"][0] for item in train_data)
eval_categories = set(item["expected"][0] for item in eval_data)

print("\n\nCategories in training set")
print(train_categories)

print("\n\nCategories in eval set")
print(eval_categories)

# 3. Baseline: Embedding Search
---

We will use `all-MiniLM-L6-v2` for our embedding model— we used the same model in our earlier lectures.

Our approach is as follows:
1. We will embed all category names into a vector database (LanceDB)
2. For each transaction, we will search the vector database for the most similar category names
3. We will measure how often we find the correct category

> 📚 **TERM: MRR (Mean Reciprocal Rank)**  
> If the correct answer is at position *k* in your ranked list, the reciprocal rank is *1/k*. MRR averages this across all queries. MRR@1 = accuracy (was the top-ranked the right one?).

> 📚 **TERM: Recall@k**  
> What fraction of the relevant items appear in your top-k results? For our task (one correct category per transaction), Recall@k = 1 if the correct category is in the top k, 0 otherwise.

In [ ]:
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def create_lancedb_table(model_name, categories, table_name):
    """Create a LanceDB table with category embeddings using the specified model."""
    model = get_registry().get("huggingface").create(name=model_name)

    class Category(LanceModel):
        text: str = model.SourceField()
        embedding: Vector(model.ndims()) = model.VectorField()

    db = lancedb.connect("../temp/lancedb")
    table = db.create_table(table_name, schema=Category, mode="overwrite")
    table.add([{"text": cat["category"]} for cat in categories])
    return table

# Create baseline table — embeds the 28 category names
baseline_table = create_lancedb_table(BASE_MODEL, categories, "categories-baseline")
print(f"Created table with {baseline_table.count_rows()} categories")

In [ ]:
def evaluate(eval_data, table, k_values=[1, 3, 5, 10], reranker=None, retrieve_k=20):
    """Evaluate retrieval performance using MRR and Recall."""
    scores = {f"mrr@{k}": [] for k in k_values}
    scores.update({f"recall@{k}": [] for k in k_values})

    for item in eval_data:
        query = item["input"]
        expected = item["expected"]

        # Retrieve candidates with bi-encoder
        results = table.search(query, query_type="vector").limit(retrieve_k).to_list()
        predictions = [r["text"] for r in results]

        # Optionally rerank with cross-encoder
        if reranker is not None:
            pairs = [[query, doc] for doc in predictions]
            rerank_scores = reranker.predict(pairs)
            ranked = [doc for _, doc in sorted(
                zip(rerank_scores, predictions), reverse=True
            )]
            predictions = ranked

        for k in k_values:
            scores[f"mrr@{k}"].append(calculate_mrr(predictions[:k], expected))
            scores[f"recall@{k}"].append(get_recall(predictions[:k], expected))

    return {metric: np.mean(vals) for metric, vals in scores.items()}

In [ ]:
# Evaluate baseline
baseline_scores = evaluate(eval_data, baseline_table)

print("Baseline Results (all-MiniLM-L6-v2):")
print("-" * 40)
for metric, score in baseline_scores.items():
    print(f"  {metric:>10s}: {score:.4f}")

The baseline gets only about 5% MRR@1 — it gets the right category on the first try just 5% of the time. The general-purpose model (`all-MiniLM-L6-v2`) doesn't understand how transaction descriptions relate to expense categories.

# 4. Fine-tuning the Embedding Model
---

> 📚 **TERM: Fine-tuning**  
> Taking a model that was pre-trained on general data and continuing to train it on your specific domain data. The model keeps its general knowledge (hopefully) but learns to better understand your domain.

Our baseline model (`all-MiniLM-L6-v2`) was trained on general web text. It doesn't know that "Tooly, Survey Software, Marketing" should map to "Subscriptions & Memberships". Fine-tuning teaches it these domain-specific relationships.

### Semi-Hard Triplet Loss

We'll use the **BatchSemiHardTripletLoss** loss function. This loss function has the following pieces:

1. An **anchor** (your main example) 
  - in our case, this will be a transaction (all information but the label)
2. A **positive** match (something similar) 
  - in our case, this is the correct category
3. A **negative** match (something different) 
  - in our case, this is the  wrong category — an incorrect category that is not quite right

The key thing here is to find **semi-hard negatives**: negatives that are not way off, but that are close to the anchor (making them hard to distinguish) but not closer than the positive. 
- *This will force our model to learn fine-grained distinctions.*

<img src="../assets/semi-hard-negative.png" width="800" />

The loss function mines these semi-hard negatives **within each training batch**, so we don't need to explicitly create triplets — we just need sentences and their category labels.

Next we will reformat our data into what `sentence-transformers` requires for `BatchSemiHardTripletLoss`

In [ ]:
# Split training data: keep 20% for monitoring during training
TEST_SIZE = 0.2
test_data = train_data[:int(len(train_data) * TEST_SIZE)]
train_split = train_data[int(len(train_data) * TEST_SIZE):]

print(f"Training examples: {len(train_split)}")
print(f"Test examples (for monitoring): {len(test_data)}")
print(f"Eval examples (for final evaluation): {len(eval_data)}")


def create_labels(data):
    """Map each category to a unique integer label."""
    categories_seen = defaultdict(list)
    for item in data:
        categories_seen[item["expected"][0]].append(item)
    return {label: idx for idx, label in enumerate(categories_seen.keys())}


def create_sentence_label_dataset(data, label_to_idx):
    """Create a HuggingFace Dataset with sentence-label pairs."""
    return Dataset.from_dict({
        "sentence": [item["input"] for item in data],
        "label": [label_to_idx[item["expected"][0]] for item in data],
    })

# Create label mapping and datasets
label_to_idx = create_labels(train_split)
train_dataset = create_sentence_label_dataset(train_split, label_to_idx)
test_dataset = create_sentence_label_dataset(test_data, label_to_idx)

print(f"\nUnique labels: {len(label_to_idx)}")
print(f"\nSample from training dataset:")
print(f"  Sentence: {train_dataset[0]['sentence'][:80]}...")
print(f"  Label: {train_dataset[0]['label']}")

Next we will create some triplets in the easiest possible way: just randomly pick a negative for each anchor/positive pair.

The triplets created are:
  - Anchor: a transaction text
  - Positive: the category name string (e.g., "Travel: Airfare")
  - Negative: a random wrong category name string

  These will be used for evaluation

In [ ]:
def create_triplets(data):
    """Create anchor/positive/negative triplets for the TripletEvaluator."""
    all_labels = set(item["expected"][0] for item in data)
    anchors, positives, negatives = [], [], []

    for item in data:
        label = item["expected"][0]
        anchors.append(item["input"])
        positives.append(label)
        negatives.append(random.choice([l for l in all_labels if l != label]))

    return anchors, positives, negatives


# Create triplets for evaluating during training
test_anchors, test_positives, test_negatives = create_triplets(test_data)

# Here we use these triplets to compute cosine accuracy: for each triplet, we will check whether the embedding of the anchor is closer to the positive than to the negative example
triplet_evaluator = TripletEvaluator(
    anchors=test_anchors,
    positives=test_positives,
    negatives=test_negatives,
    name="embedding-eval",
)

# Check baseline triplet accuracy before fine-tuning
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
baseline_triplet_results = triplet_evaluator(model)
print(f"Baseline triplet accuracy: {baseline_triplet_results['embedding-eval_cosine_accuracy']:.4f}")

We have a pretty high baseline accuracy. This means that the off-the-shelf MiniLM model already gets ~89% of random triplets right before any fine-tuning. But that's the correct category VS one random wrong category... when we compare with many, an incorrect prediction is much more likely.

Now it's time to fine-tune the model using the `BatchSemiHardTripletLoss` loss function.

 The actual training doesn't use explicit triplets at all. `BatchSemiHardTripletLoss` mines them on-the-fly from each batch, and those are all transaction-to-transaction:
  - Anchor: a transaction
  - Positive: another transaction with the same category label
  - Negative: a transaction with a different category label that's close in embedding space (semi-hard)

Training optimizes transaction-vs-transaction similarity, but evaluation (and the actual downstream task) measures transaction-vs-category-name similarity. This works because clustering transactions by category indirectly improves matching against category names — but we're not directly optimizing for that.

In [ ]:
# Set up fine-tuning
loss = BatchSemiHardTripletLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="../temp/ft-embedding-checkpoints",
    num_train_epochs=10, # number of full passes through the training data
    per_device_train_batch_size=16, # how many examples per training batch
    per_device_eval_batch_size=16, # same but for evaluation
    learning_rate=2e-5, # small enough to not destroy pre-trained weights, large enough to learn
    warmup_ratio=0.1, # first 10% of training steps gradually ramp up the LR from 0
    batch_sampler=BatchSamplers.NO_DUPLICATES, # ensures no two examples in a batch share the same label.
    eval_strategy="epoch", # run evaluation after each epoch
    save_strategy="epoch", # save a checkpoint after each epoch
    save_total_limit=1, # only keep the most recent checkpoint
    logging_steps=10, # print training loss every 10 steps
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss=loss,
    evaluator=triplet_evaluator,
)

# Train!
trainer.train()

# Save the fine-tuned model
model.save_pretrained("../temp/ft-embedding-model")
print("\nFine-tuned model saved to ../temp/ft-embedding-model")

The columns above::
  - **Training Loss**: this is the semi-hard triplet loss on training batches. Lower = the model is getting better at pushing same-category transactions together and different-category ones apart. It drops from 4.66 to 4.27 — which is a good improvement.
  - **Validation Loss** same metric on the held-out 20% split. Tracks training loss closely (4.71 → 4.34), which
   means the model is not overfitting — it's generalizing well.
  - **Embedding-eval Cosine Accuracy**: the triplet evaluator from earlier. For each (anchor, positive, negative) triplet, is the anchor closer to the positive than the negative? Goes from 0.93 → 0.97. Remember the baseline before fine-tuning was 0.89, so the model improved meaningfully.

  Some key takeaways from this table:

  1. **Both losses are still decreasing (at epoch 10)** — you could train longer and probably squeeze out a bit
  more
  2. **No overfitting**: validation loss tracks training loss, no divergence
  3. **The cosine accuracy plateaus around epoch 5-6 (~0.965)** — but that's partly because random negatives are easy. The real gains show up in the downstream MRR/Recall evaluation
  4. **38 seconds total for 10 epochs** — this is a tiny model (22M params), very fast to fine-tune

Note:
  - If you increase the number of epochs a lot, you'll start seeing NaN's. This means that the model collapsed. Training loss 0.0, cosine accuracy 0.0, validation NaN — this means all embeddings have become identical (or zero), so there are no distances to compute and everything breaks.

In [ ]:
# Check triplet accuracy after fine-tuning
ft_triplet_results = triplet_evaluator(model)
print(f"Baseline triplet accuracy:   {baseline_triplet_results['embedding-eval_cosine_accuracy']:.4f}")
print(f"Fine-tuned triplet accuracy: {ft_triplet_results['embedding-eval_cosine_accuracy']:.4f}")

# Create LanceDB table with fine-tuned model and evaluate
ft_table = create_lancedb_table(
    "../temp/ft-embedding-model", categories, "categories-finetuned"
)
ft_scores = evaluate(eval_data, ft_table)

print(f"\nFine-tuned Embedding Results:")
print("-" * 40)
for metric, score in ft_scores.items():
    print(f"  {metric:>10s}: {score:.4f}")

In [ ]:
# Compare baseline vs fine-tuned embeddings
comparison_df = pd.DataFrame({
    "Baseline": baseline_scores,
    "Fine-tuned": ft_scores,
}).T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MRR comparison
mrr_cols = [c for c in comparison_df.columns if c.startswith("mrr")]
x = np.arange(len(mrr_cols))
width = 0.3

ax1.bar(x - width/2, comparison_df.loc["Baseline", mrr_cols], width, label="Baseline", color="#4C72B0")
ax1.bar(x + width/2, comparison_df.loc["Fine-tuned", mrr_cols], width, label="Fine-tuned", color="#55A868")
ax1.set_title("Mean Reciprocal Rank (MRR)")
ax1.set_xticks(x)
ax1.set_xticklabels(mrr_cols)
ax1.set_ylabel("Score")
ax1.set_ylim(0, 1)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Recall comparison
recall_cols = [c for c in comparison_df.columns if c.startswith("recall")]
x = np.arange(len(recall_cols))

ax2.bar(x - width/2, comparison_df.loc["Baseline", recall_cols], width, label="Baseline", color="#4C72B0")
ax2.bar(x + width/2, comparison_df.loc["Fine-tuned", recall_cols], width, label="Fine-tuned", color="#55A868")
ax2.set_title("Recall")
ax2.set_xticks(x)
ax2.set_xticklabels(recall_cols)
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print improvement
print("\nImprovement from fine-tuning:")
for metric in comparison_df.columns:
    base = comparison_df.loc["Baseline", metric]
    ft = comparison_df.loc["Fine-tuned", metric]
    pct = (ft - base) / base * 100 if base > 0 else 0
    print(f"  {metric:>10s}: {base:.4f} -> {ft:.4f} ({pct:+.1f}%)")

Fine-tuning took the model from 5% to ~45% MRR@1 — a dramatic improvement with just ~200 training examples and a few minutes of training. The model now understands transaction-to-category relationships much better. 

# 5. Rerankers
---

So far, we've used  embeddings -- **bi-encoders**.
- Embeddings are a model that independently converts queries, documents, etc into vectors, then compares them by cosine similarity. 
- This is nice and fast because documents can be pre-embedded once, and then we just compare the vector similarity.

A **reranker / cross-encoder** takes a different approach 
— it processes the query and document **together** as a single input
- Because the cross-encoder sees both texts at once, it can capture interactions between them that a bi-encoder misses. 
- But it's much slower — it must score each (query, document) pair individually.

This leads to the **retrieve-and-rerank** pipeline:

```text
All documents   --> [Bi-encoder] --> Top K candidates --> [Cross-encoder] --> Re-scored top results
     (fast)            (1000s)          (slow)                (K)               (precise)
```

This gives you the speed of bi-encoders with the accuracy of cross-encoders.

# 6. Using a Pre-trained Reranker
---

Let's try `cross-encoder/ms-marco-MiniLM-L-6-v2` — a popular reranker trained on the **MS MARCO** web search dataset (millions of search query + web passage pairs).

**The question**: Will a reranker trained on web search help with transaction categorization?

In [ ]:
# Load pretrained cross-encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Demo: score a single transaction against a few categories
sample_query = eval_data[0]["input"]
sample_expected = eval_data[0]["expected"][0]

print(f"Transaction:{sample_query}")
print(f"Correct category: {sample_expected}\n")

# Score against a few categories
demo_categories = [sample_expected] + random.sample(
    [cat["category"] for cat in categories if cat["category"] != sample_expected], k=4
)

pairs = [[sample_query, cat] for cat in demo_categories]
scores = reranker.predict(pairs)

print("Cross-encoder scores:")
for cat, score in sorted(zip(demo_categories, scores), key=lambda x: x[1], reverse=True):
    marker = "  <-- correct" if cat == sample_expected else ""
    print(f"  {score:+.4f}  {cat}{marker}")

In [ ]:
# Evaluate: retrieve with baseline bi-encoder, then rerank
pretrained_reranker_scores = evaluate(
    eval_data, baseline_table, reranker=reranker, retrieve_k=20
)

print("Pretrained Reranker Results (baseline embeddings + reranker):")
print("-" * 55)
print(f"{'Metric':>13s}  {'Baseline':>10s}  {'+ Reranker':>10s}  {'Change':>10s}")
print("-" * 55)
for metric in pretrained_reranker_scores:
    base = baseline_scores[metric]
    rerank = pretrained_reranker_scores[metric]
    change = rerank - base
    print(f"  {metric:>10s}  {base:>10.4f}  {rerank:>10.4f}  {change:>+10.4f}")

### The pretrained reranker is limited

The pretrained reranker helps somewhat compared to the raw baseline (which was only 5% MRR@1). But notice two things:

1. **It can't match fine-tuned embeddings** — the fine-tuned bi-encoder alone (MRR@1 ~39%) beats the pretrained reranker (~26%). Domain-specific fine-tuning outperforms a generic reranker.

2. **The scores are all negative** — the cross-encoder was trained on MS MARCO (web search: short queries, long passages). Our task is the opposite (long transactions, short category names). The model doesn't really "understand" our domain.

**Lesson**: A pretrained reranker is a quick experiment to try, but it can't replace domain-specific training. Let's see what happens when we fine-tune the reranker on our data.

# 7. Fine-tuning the Reranker
---


Since the pretrained reranker doesn't understand our domain, let's fine-tune it. We need training data with:
- **Positive pairs**: (transaction, correct category) with label 1
- **Negative pairs**: (transaction, wrong category) with label 0


We'll randomly sample 4 wrong categories per transaction as negatives.

In [ ]:
# Get all category names
all_categories = [cat["category"] for cat in categories]

# Create training pairs
sentences1 = []  # transaction texts
sentences2 = []  # category names
labels = []      # 1 for correct, 0 for incorrect

for item in train_data:
    query = item["input"]
    correct_category = item["expected"][0]

    # Positive pair
    sentences1.append(query)
    sentences2.append(correct_category)
    labels.append(1)

    # Negative pairs: 4 random wrong categories
    wrong = random.sample(
        [c for c in all_categories if c != correct_category], k=4
    )
    for neg in wrong:
        sentences1.append(query)
        sentences2.append(neg)
        labels.append(0)

reranker_dataset = Dataset.from_dict({
    "sentence1": sentences1,
    "sentence2": sentences2,
    "label": labels,
})

print(f"Total training pairs: {len(reranker_dataset)}")
print(f"  Positive pairs: {sum(1 for l in labels if l == 1)}")
print(f"  Negative pairs: {sum(1 for l in labels if l == 0)}")
print(f"  Ratio: 1:{sum(1 for l in labels if l == 0) / sum(1 for l in labels if l == 1):.0f}")

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

# Initialize a fresh cross-encoder for fine-tuning
ft_reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

loss = BinaryCrossEntropyLoss(ft_reranker)

args = CrossEncoderTrainingArguments(
    output_dir="../temp/ft-reranker-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=50,
)

reranker_trainer = CrossEncoderTrainer(
    model=ft_reranker,
    args=args,
    train_dataset=reranker_dataset,
    loss=loss,
)

reranker_trainer.train()

# Save the fine-tuned reranker
ft_reranker.save_pretrained("../temp/ft-reranker-model")
print("\nFine-tuned reranker saved to ../temp/ft-reranker-model")

In [ ]:
# Load the fine-tuned reranker and evaluate
ft_reranker_loaded = CrossEncoder("../temp/ft-reranker-model")

ft_reranker_scores = evaluate(
    eval_data, baseline_table, reranker=ft_reranker_loaded, retrieve_k=20
)

print("Fine-tuned Reranker Results (baseline embeddings + fine-tuned reranker):")
print("-" * 55)
print(f"{'Metric':>13s}  {'Baseline':>11s}  {'Pretrained':>12s}  {'Fine-tuned':>10s}")
print("-" * 55)
for metric in ft_reranker_scores:
    base = baseline_scores[metric]
    pre = pretrained_reranker_scores[metric]
    ft = ft_reranker_scores[metric]
    print(f"  {metric:>10s}  {base:>10.4f}  {pre:>10.4f}  {ft:>10.4f}")

Domain-specific training data makes all the difference — the model now understands what makes a transaction match a category.

---

# 8. Putting It All Together

Let's compare all our approaches, including the combination of fine-tuned embeddings with the fine-tuned reranker.

In [ ]:
# Evaluate the best combo: fine-tuned embeddings + fine-tuned reranker
combo_scores = evaluate(
    eval_data, ft_table, reranker=ft_reranker_loaded, retrieve_k=20
)

# Build comparison table
all_results = pd.DataFrame({
    "Embeddings": baseline_scores,
    "Embeddings + Reranker": pretrained_reranker_scores,
    "Embeddings + FT Reranker": ft_reranker_scores,
    "FT Embeddings": ft_scores,
    "FT Embeddings + FT Reranker ": combo_scores,
}).T

print("\nFull Comparison:")
print(all_results.round(4).to_string())

In [ ]:
# Final comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

# MRR comparison
mrr_cols = [c for c in all_results.columns if c.startswith("mrr")]
x = np.arange(len(mrr_cols))
width = 0.15

for i, (name, row) in enumerate(all_results.iterrows()):
    offset = (i - 2) * width
    ax1.bar(x + offset, row[mrr_cols], width, label=name, color=colors[i])

ax1.set_title("Mean Reciprocal Rank (MRR)", fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(mrr_cols)
ax1.set_ylabel("Score")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Recall comparison
recall_cols = [c for c in all_results.columns if c.startswith("recall")]
x = np.arange(len(recall_cols))

for i, (name, row) in enumerate(all_results.iterrows()):
    offset = (i - 2) * width
    ax2.bar(x + offset, row[recall_cols], width, label=name, color=colors[i])

ax2.set_title("Recall", fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels(recall_cols)
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

# Summary

| Technique | What it does | When to use it |
|---|---|---|
| **Fine-tune embeddings** | Adapts the bi-encoder to your domain | When you have domain-specific labeled data and want better recall |
| **Pretrained reranker** | Adds a second-stage scoring model | When your domain is similar to web search (the training data of most rerankers) |
| **Fine-tune reranker** | Adapts the cross-encoder to your domain | When pretrained rerankers are limited for your domain (like here!) |
| **Stack both** | Fine-tune embeddings + fine-tune reranker | When you want the best possible performance and have training data |

### Key takeaways

1. **Always measure first** — the pretrained reranker helped a little, but couldn't match even basic embedding fine-tuning. Without evaluation, you might have stopped there.
2. **Domain-specific fine-tuning is powerful** — even with just ~260 training examples, fine-tuning took us from 5% to 55% MRR@1.
3. **Improvements stack** — fine-tuned embeddings improve recall, fine-tuned rerankers improve precision, and combining them gives the best results (~96% Recall@5).
4. **The retrieve-and-rerank pattern** is a fundamental building block — fast retrieval for recall, precise reranking for precision.

# 9. Bonus: Can Off-the-Shelf APIs Beat Fine-tuning?
---

We showed that fine-tuning small open-source models produces dramatic improvements. But what if you just threw money at the problem — using the best available commercial embeddings and rerankers?

Let's compare our fine-tuned pipeline against an **off-the-shelf premium stack**:
- **OpenAI `text-embedding-3-large`** — one of the best commercial embedding models
- **Cohere `rerank-v3.5`** — a state-of-the-art commercial reranker

> **Note**: This section requires `OPENAI_API_KEY` and `COHERE_API_KEY` in your `.env` file. If you don't have a Cohere key, you can skip this section — the main notebook is complete without it.

In [ ]:
import cohere

# OpenAI embeddings: create LanceDB table with text-embedding-3-large
openai_model = get_registry().get("openai").create(name="text-embedding-3-large")

class OpenAICategory(LanceModel):
    text: str = openai_model.SourceField()
    embedding: Vector(openai_model.ndims()) = openai_model.VectorField()

db = lancedb.connect("../temp/lancedb")
openai_table = db.create_table("categories-openai-large", schema=OpenAICategory, mode="overwrite")
openai_table.add([{"text": cat["category"]} for cat in categories])

print(f"Created OpenAI table with {openai_table.count_rows()} categories (dim={openai_model.ndims()})")

# Cohere reranker
co = cohere.Client()

def cohere_rerank(query, documents, top_n=None):
    """Rerank documents using Cohere's rerank API."""
    response = co.rerank(
        model="rerank-v3.5",
        query=query,
        documents=documents,
        top_n=top_n or len(documents),
    )
    return [documents[r.index] for r in response.results]


def evaluate_with_cohere(eval_data, table, k_values=[1, 3, 5, 10], retrieve_k=20):
    """Evaluate retrieval with Cohere reranking."""
    scores = {f"mrr@{k}": [] for k in k_values}
    scores.update({f"recall@{k}": [] for k in k_values})

    for item in eval_data:
        query = item["input"]
        expected = item["expected"]

        results = table.search(query, query_type="vector").limit(retrieve_k).to_list()
        predictions = [r["text"] for r in results]

        # Rerank with Cohere
        predictions = cohere_rerank(query, predictions)

        for k in k_values:
            scores[f"mrr@{k}"].append(calculate_mrr(predictions[:k], expected))
            scores[f"recall@{k}"].append(get_recall(predictions[:k], expected))

    return {metric: np.mean(vals) for metric, vals in scores.items()}

print("Cohere client ready.")

In [ ]:
# Evaluate off-the-shelf premium stack
openai_scores = evaluate(eval_data, openai_table)
openai_cohere_scores = evaluate_with_cohere(eval_data, openai_table)

# Side-by-side comparison: premium APIs vs fine-tuned
comparison = pd.DataFrame({
    "FT MiniLM + FT Reranker": combo_scores,
    "OpenAI Large": openai_scores,
    "OpenAI Large + Cohere": openai_cohere_scores,
}).T

print("Off-the-Shelf vs Fine-Tuned:")
print(comparison.round(4).to_string())

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#55A868", "#CCB974", "#C44E52"]

mrr_cols = [c for c in comparison.columns if c.startswith("mrr")]
x = np.arange(len(mrr_cols))
width = 0.25

for i, (name, row) in enumerate(comparison.iterrows()):
    offset = (i - 1) * width
    ax1.bar(x + offset, row[mrr_cols], width, label=name, color=colors[i])

ax1.set_title("MRR: Fine-tuned vs Off-the-Shelf", fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(mrr_cols)
ax1.set_ylabel("Score")
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

recall_cols = [c for c in comparison.columns if c.startswith("recall")]
x = np.arange(len(recall_cols))

for i, (name, row) in enumerate(comparison.iterrows()):
    offset = (i - 1) * width
    ax2.bar(x + offset, row[recall_cols], width, label=name, color=colors[i])

ax2.set_title("Recall: Fine-tuned vs Off-the-Shelf", fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels(recall_cols)
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Exercise: Fine-tune Search for Wayfair Products
---

In Lecture 4, we built a product search system for the [WANDS dataset](https://github.com/wayfair/WANDS) using off-the-shelf embeddings. Your task today is to improve that system using fine-tuning.

- You already have the data in `data/`: 
  - `wayfair-products.csv` — ~43,000 products with names, descriptions, categories, and features
  - `wayfair-queries.csv` — 480 search queries
  - `wayfair-labels.csv` — 233,000 human relevance judgments (Exact / Partial / Irrelevant)
  - This is a much larger and more realistic dataset than the Ramp transactions we used in this lecture.

Your task is the following
1. **Establish a baseline** — This should be the work that you've already done in Lecture 4. Use Precision@K, Recall@K or NDCG@K. Remember that you can compute these metrics using the provided labels, and that some helpful functions live in `helpers.py`.
2. **Fine-tune an embedding model** — Adapt the techniques from this lecture to the WANDS data. This is where you need to think carefully about data formatting (see tradeoffs below).
3. **Fine-tune a reranker** — Create (query, product, label) training pairs and fine-tune a cross-encoder.
4. **Evaluate and compare** — Compare the performance of the baseline system against the fine-tuned embeddings, the fine-tuned reranker, and the the full fine-tunedpipeline.

### Key tradeoffs to consider

**(1) How will you create training data for the embedding model?**

The Ramp task had a clean category label per transaction. WANDS has (query, product, relevance) triples — a different structure.
- **Option A**: Treat each query as a "class" — all products labeled Exact or Partial for that query belong to the same class. Then use `BatchSemiHardTripletLoss` as we did in the lecture. Simple, but queries with few relevant products may not work well.
- **Option B**: Use a **pair-based loss** (e.g., `CosineSimilarityLoss` or `ContrastiveLoss`) where each (query, product) pair gets a similarity score derived from the label (Exact=1.0, Partial=0.5, Irrelevant=0.0).
- **Option C**: Construct explicit triplets — (query, relevant product, irrelevant product) — and use `TripletLoss` directly.

**(2) What text will you embed for products?**
Again, you have many options here.
- Just the product name? Name + category? Name + description? The full feature text?
- Longer text gives the model more signal but increases noise. Short text is cleaner but may miss important details.
- You could also concatenate fields: `f"{product_name} | {category_hierarchy}"`.

**(3) How many negatives for the reranker?**
- In the lecture we used 4 random negatives per positive. With 43k products, random negatives will almost always be easy.
- Hard negatives (products that are close in embedding space but irrelevant) will be much more effective. You could use your baseline embeddings to find these.
- More negatives = more training data but slower training. Fewer but harder negatives may be more efficient.

**(4) How much data will you use?**
- You have 480 queries with labels. That's enough for fine-tuning, but you'll want to split into train/eval.
- Not all queries have the same number of labeled products. Some queries have hundreds of judgments, others have few.
- You could also augment your data with LLM-generated synthetic queries for products -- similarly to what we did in this Lecture.